In [1]:
!pip install -q kaggle

In [2]:
from google.colab import userdata

KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
KAGGLE_KEY = userdata.get("KAGGLE_KEY")

print("Kaggle credentials loaded")

Kaggle credentials loaded


In [3]:
import os

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

print("Kaggle configured")

Kaggle configured


In [4]:
!kaggle datasets download -d svizor/retail-sales-forecasting-data

Dataset URL: https://www.kaggle.com/datasets/svizor/retail-sales-forecasting-data
License(s): CC-BY-NC-SA-4.0
100% 211M/211M [00:02<00:00, 103MB/s]



In [5]:
import zipfile

zip_path = "/content/retail-sales-forecasting-data.zip"
extract_path = "/content/retail_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted")

Dataset extracted


In [6]:
import glob

files = glob.glob(
    "/content/retail_data/**/*",
    recursive=True
)

for file in files:
    print(file)

/content/retail_data/sales.csv
/content/retail_data/stores.csv
/content/retail_data/actual_matrix.csv
/content/retail_data/markdowns.csv
/content/retail_data/discounts_history.csv
/content/retail_data/price_history.csv
/content/retail_data/catalog.csv
/content/retail_data/online.csv


In [7]:
import pandas as pd
import numpy as np

sales = pd.read_csv(
    "/content/retail_data/sales.csv"
)

stores = pd.read_csv(
    "/content/retail_data/stores.csv"
)

catalog = pd.read_csv(
    "/content/retail_data/catalog.csv"
)

discounts = pd.read_csv(
    "/content/retail_data/discounts_history.csv"
)

markdowns = pd.read_csv(
    "/content/retail_data/markdowns.csv"
)

price_history = pd.read_csv(
    "/content/retail_data/price_history.csv"
)

print("Sales:", sales.shape)
print("Stores:", stores.shape)
print("Catalog:", catalog.shape)
print("Discounts:", discounts.shape)
print("Markdowns:", markdowns.shape)
print("Price history:", price_history.shape)

Sales: (7432685, 7)
Stores: (4, 6)
Catalog: (219810, 9)
Discounts: (3746744, 9)
Markdowns: (8979, 7)
Price history: (698626, 6)


In [10]:
print(sales_model.columns.tolist())
print(price_history.columns.tolist())

['Unnamed: 0_x', 'date', 'item_id', 'quantity', 'price_base', 'sum_total', 'store_id', 'Unnamed: 0_y', 'division', 'format', 'city', 'area', 'Unnamed: 0', 'dept_name', 'class_name', 'subclass_name', 'item_type', 'weight_volume', 'weight_netto', 'fatness']
['Unnamed: 0', 'date', 'item_id', 'price', 'code', 'store_id']


In [11]:
def drop_unnamed(df):
    cols_to_drop = [c for c in df.columns if c.startswith("Unnamed")]
    return df.drop(columns=cols_to_drop, errors="ignore")

sales = drop_unnamed(sales)
stores = drop_unnamed(stores)
catalog = drop_unnamed(catalog)
price_history = drop_unnamed(price_history)
discounts = drop_unnamed(discounts)
markdowns = drop_unnamed(markdowns)

In [12]:
# 1. Start from sales (the core transaction table)
sales_model = sales.copy()

# 2. Merge store attributes (store_id is the join key)
sales_model = sales_model.merge(stores, on="store_id", how="left")

# 3. Merge product/catalog attributes (item_id is the join key)
sales_model = sales_model.merge(catalog, on="item_id", how="left")

# 4. Merge price history (item_id + store_id + date, adjust keys if needed)
sales_model = sales_model.merge(
    price_history, on=["item_id", "store_id", "date"], how="left"
)

# 5. Merge discount history
sales_model = sales_model.merge(
    discounts, on=["item_id", "store_id", "date"], how="left"
)

# 6. Merge markdowns
sales_model = sales_model.merge(
    markdowns, on=["item_id", "store_id", "date"], how="left"
)

# 7. Feature-engineer date parts (since raw 'date' isn't usable directly)
sales_model["date"] = pd.to_datetime(sales_model["date"])
sales_model["year"] = sales_model["date"].dt.year
sales_model["month"] = sales_model["date"].dt.month
sales_model["day"] = sales_model["date"].dt.day
sales_model["day_of_week"] = sales_model["date"].dt.dayofweek
sales_model["week_of_year"] = sales_model["date"].dt.isocalendar().week
sales_model["quarter"] = sales_model["date"].dt.quarter
sales_model["is_weekend"] = sales_model["day_of_week"].isin([5, 6]).astype(int)

print("Merged shape:", sales_model.shape)
print("Columns:", sales_model.columns.tolist())
print("Missing values per column:")
print(sales_model.isnull().sum())

Merged shape: (7463395, 34)
Columns: ['date', 'item_id', 'quantity_x', 'price_base', 'sum_total', 'store_id', 'division', 'format', 'city', 'area', 'dept_name', 'class_name', 'subclass_name', 'item_type', 'weight_volume', 'weight_netto', 'fatness', 'price_x', 'code', 'sale_price_before_promo', 'sale_price_time_promo', 'promo_type_code', 'doc_id', 'number_disc_day', 'normal_price', 'price_y', 'quantity_y', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend']
Missing values per column:
date                             0
item_id                          0
quantity_x                       0
price_base                       0
sum_total                        0
store_id                         0
division                         0
format                           0
city                             0
area                             0
dept_name                    36702
class_name                   36702
subclass_name                36702
item_type                   6

In [13]:
# 1. Resolve the quantity duplicate — quantity_x is the real sales quantity (0 missing)
sales_model = sales_model.rename(columns={"quantity_x": "quantity"})
sales_model = sales_model.drop(columns=["quantity_y"])  # 99.9% missing, from markdowns, not usable

# 2. Resolve the price duplicate — keep price_base (0 missing), drop the near-empty ones
sales_model = sales_model.drop(columns=["price_x", "price_y", "code"])  # too sparse to use

# 3. Drop other columns that are mostly empty and not critical
sales_model = sales_model.drop(columns=["normal_price"])  # 99.9% missing

# 4. For promo/discount columns — missing likely means "no promo that day", so fill with 0/None instead of dropping
promo_cols = ["sale_price_before_promo", "sale_price_time_promo", "promo_type_code", "doc_id", "number_disc_day"]
for col in promo_cols:
    if sales_model[col].dtype == "object":
        sales_model[col] = sales_model[col].fillna("None")
    else:
        sales_model[col] = sales_model[col].fillna(0)

# 5. For product attributes with moderate missingness, fill with a placeholder
product_cols = ["dept_name", "class_name", "subclass_name", "item_type"]
for col in product_cols:
    sales_model[col] = sales_model[col].fillna("Unknown")

# 6. weight/fatness — numeric, fill with median (or leave as NaN if your pipeline's imputer handles it)
for col in ["weight_volume", "weight_netto", "fatness"]:
    sales_model[col] = sales_model[col].fillna(sales_model[col].median())

print("Final shape:", sales_model.shape)
print("Remaining missing values:")
print(sales_model.isnull().sum())

Final shape: (7463395, 29)
Remaining missing values:
date                       0
item_id                    0
quantity                   0
price_base                 0
sum_total                  0
store_id                   0
division                   0
format                     0
city                       0
area                       0
dept_name                  0
class_name                 0
subclass_name              0
item_type                  0
weight_volume              0
weight_netto               0
fatness                    0
sale_price_before_promo    0
sale_price_time_promo      0
promo_type_code            0
doc_id                     0
number_disc_day            0
year                       0
month                      0
day                        0
day_of_week                0
week_of_year               0
quarter                    0
is_weekend                 0
dtype: int64


In [14]:
X = sales_model.drop(columns=["quantity", "sum_total", "date"]).copy()
y = sales_model["quantity"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nX columns:", X.columns.tolist())

X shape: (7463395, 26)
y shape: (7463395,)

X columns: ['item_id', 'price_base', 'store_id', 'division', 'format', 'city', 'area', 'dept_name', 'class_name', 'subclass_name', 'item_type', 'weight_volume', 'weight_netto', 'fatness', 'sale_price_before_promo', 'sale_price_time_promo', 'promo_type_code', 'doc_id', 'number_disc_day', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend']


In [15]:
pd.set_option('display.max_columns', None)

for col in X.columns:
    print(f"\n--- {col} ({X[col].dtype}) ---")
    if X[col].dtype == 'object':
        print(f"Unique values ({X[col].nunique()} total):")
        print(X[col].unique()[:25])
    else:
        print(f"Range: {X[col].min()} to {X[col].max()}")


--- item_id (object) ---
Unique values (28182 total):
['293375605257' 'a66fdf2c0ae7' 'daa46ef49b7a' 'a3b49c1bf758'
 'ab611c5cef62' 'be3b7250209f' '651c34ed4958' '7390a33d7f34'
 'd2f975315972' 'd788cc0f8e2a' 'f0309b5a974b' '0aa837b93d7e'
 '8fd19004e664' '5f3114373ddd' '393689c97532' '668df13645a2'
 '68b1397119e1' '8b2e530f97ab' '56edb7bdecaf' '0b211fdb94a9'
 '8c9d638a9581' 'bb71024009cb' '6a36bad625b6' 'd862997509e4'
 '6f9d7773db65']

--- price_base (float64) ---
Range: -21676.67 to 28999.9

--- store_id (int64) ---
Range: 1 to 4

--- division (object) ---
Unique values (2 total):
['Div1' 'Div2']

--- format (object) ---
Unique values (4 total):
['Format-1' 'Format-6' 'Format-7 express' 'MaxiEuro']

--- city (object) ---
Unique values (3 total):
['City1' 'City2' 'City3']

--- area (int64) ---
Range: 109 to 1887

--- dept_name (object) ---
Unique values (183 total):
['ПРИКАССА' 'ЛИМОНАДЫ' 'СЛОЙКА' 'ПЕРСОНАЛЬНЫЙ УХОД'
 'СОВРЕМЕННАЯ МОЛОЧНАЯ КАТЕГОРИЯ' 'СПЕЦИИ,ПРИПРАВА' 'ЧИПСЫ'
 'КОРЖИ И 

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5970716, 26)
X_test: (1492679, 26)
y_train: (5970716,)
y_test: (1492679,)


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identify categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['item_id', 'division', 'format', 'city', 'dept_name', 'class_name', 'subclass_name', 'item_type', 'doc_id']
Numerical columns: ['price_base', 'store_id', 'area', 'weight_volume', 'weight_netto', 'fatness', 'sale_price_before_promo', 'sale_price_time_promo', 'promo_type_code', 'number_disc_day', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend']


In [19]:
X_train = X_train.drop(columns=["item_id", "doc_id"])
X_test = X_test.drop(columns=["item_id", "doc_id"])

categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

Categorical: ['division', 'format', 'city', 'dept_name', 'class_name', 'subclass_name', 'item_type']
Numerical: ['price_base', 'store_id', 'area', 'weight_volume', 'weight_netto', 'fatness', 'sale_price_before_promo', 'sale_price_time_promo', 'promo_type_code', 'number_disc_day', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend']


In [21]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 13.6 MB/s eta 0:00:00


In [22]:
from catboost import CatBoostRegressor

cat_features = [
    "division",
    "format",
    "city",
    "dept_name",
    "class_name",
    "subclass_name",
    "item_type"
]

model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100,
    task_type="GPU"
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50
)

0:	learn: 26.3100613	test: 26.0923959	best: 26.0923959 (0)	total: 1.05s	remaining: 8m 44s
100:	learn: 17.4868416	test: 17.4100673	best: 17.4100673 (100)	total: 2m 14s	remaining: 8m 50s
200:	learn: 16.7266248	test: 16.9393825	best: 16.9379656 (199)	total: 4m 29s	remaining: 6m 41s
300:	learn: 16.2681382	test: 16.7808705	best: 16.7778464 (297)	total: 6m 43s	remaining: 4m 26s
400:	learn: 15.8628470	test: 16.7018857	best: 16.7018857 (400)	total: 8m 56s	remaining: 2m 12s
bestTest = 16.66830917
bestIteration = 431
Shrink model to first 432 iterations.


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=500, learning_rate=0.1, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100)

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 3.415188192677577
RMSE: 16.668316057917703
R²  : 0.6246795902884946


In [24]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print(results.head(20))

    Actual  Predicted
0   18.000  10.000303
1    1.000   2.819893
2    3.755  14.037498
3    1.000   1.631536
4    3.000   2.698975
5    1.000   1.759079
6    1.000   1.690277
7    1.000   2.318711
8   27.000  14.740717
9    5.000   6.194940
10   0.328   0.855779
11   8.000   8.343388
12   2.000   3.754110
13   3.000   2.736960
14  11.000  10.012533
15   1.000   3.074534
16   1.000   2.513923
17   5.000  11.527306
18   0.142   1.018417
19   1.000   2.670018


In [25]:
model.save_model("quantity_prediction_model.cbm")